# Stage-1 cross-chip edgeR evidence closeout

This report reuses frozen K=6 factor-family edgeR outputs. It does not rerun RCTD, NMF, pseudobulk construction, or edgeR. The joint family model remains primary; chip-wise meta-analysis is a replication and heterogeneity sensitivity layer.

Render command: `bash scripts/1116_render_crosschip_edger_closeout.sh`

In [ ]:
from pathlib import Path
import json
import platform
import pandas as pd
from IPython.display import Image, Markdown, display

project_root = Path.cwd()
if project_root.name == 'scripts':
    project_root = project_root.parent
result_dir = project_root / 'results/reanalysis/CROSS_CHIP_EDGER_STAGE1_CLOSEOUT_20261014_065410'
assert (result_dir / 'ANALYSIS_COMPLETE.ok').exists()
closeout = pd.read_csv(result_dir / '02_CANDIDATE_EVIDENCE_CLOSEOUT.tsv', sep='\t')
meta_sig = pd.read_csv(result_dir / '03_META_SIGNIFICANT_SENSITIVITY.tsv', sep='\t')
qa = pd.read_csv(result_dir / '05_QA.tsv', sep='\t')
assert qa['passed'].astype(str).str.lower().eq('true').all()
display(Markdown(f'**Final result directory:** `{result_dir}`  \n**Candidate signals:** {len(closeout)}  \n**Complete coding-RNA meta tests with global FDR < 0.05:** {len(meta_sig)}'))

## Methods and interpretation boundaries

- Primary endpoint: paired high-minus-matched-control regional expression difference.
- Biological replicate: patient.
- Primary inference: existing joint family edgeR model.
- Sensitivity inference: fixed-effect and DerSimonian-Laird random-effect meta-analysis of mapped per-chip factors.
- Approximate chip-level standard errors were recovered from the saved one-degree-of-freedom QL-F statistic.
- With only two or three chips, heterogeneity and random-effect estimates are descriptive and are not standalone exclusion rules.

In [ ]:
display(Markdown('## Quality assurance'))
display(qa)

In [ ]:
joint_n = int(closeout['joint_primary_supported'].sum())
fixed_n = int(closeout['fixed_meta_sensitivity_supported'].sum())
direction_n = int(closeout['all_observed_directionally_compatible'].sum())
independent_all_chip_n = int(((closeout['observed_chip_n'] == closeout['mapped_chip_n']) & (closeout['chip_global_fdr_n'] == closeout['mapped_chip_n'])).sum())
tier_counts = closeout['evidence_tier'].value_counts()
summary = '\n'.join([
    '## Evidence overview',
    '',
    f'- Existing candidate signals: **{len(closeout)}**.',
    f'- Joint family global coding FDR support: **{joint_n}**.',
    f'- Fixed-effect meta global FDR support: **{fixed_n}**.',
    f'- Directionally compatible across all estimable mapped chips: **{direction_n}/{len(closeout)}**.',
    f'- Independently global-FDR significant in every mapped chip: **{independent_all_chip_n}**.',
    '',
    'Evidence tiers are descriptive summaries, not new exclusion gates.'
])
display(Markdown(summary))
display(tier_counts.rename_axis('evidence_tier').reset_index(name='candidate_n'))

In [ ]:
display(Markdown('## Joint model and chip-wise meta-analysis'))
display(Image(filename=str(result_dir / 'figures/01_JOINT_VS_FIXED_META.png')))
display(Markdown('Fixed-effect meta estimates generally track the joint family estimates. Color indicates I²; high I² marks chip-effect heterogeneity rather than invalidating a factor family.'))

In [ ]:
display(Markdown('## Chip-specific effects and confidence intervals'))
display(Image(filename=str(result_dir / 'figures/02_CANDIDATE_CHIP_META_FOREST.png')))

In [ ]:
columns = [
    'family_label', 'comparison', 'gene_symbol', 'joint_log2_fc',
    'joint_global_coding_fdr', 'fixed_log2_fc', 'fixed_global_meta_fdr',
    'i2_percent', 'patient_loo_instability_flag', 'evidence_tier'
]
display(Markdown('## Unified candidate evidence table'))
display(closeout[columns].round(4))

In [ ]:
def gene_line(gene):
    row = closeout.loc[closeout['gene_symbol'].eq(gene)].iloc[0]
    direction = 'increased' if row['joint_log2_fc'] > 0 else 'decreased'
    return f"- **{gene}** was {direction} for {row['comparison'].replace('_vs_', ' vs ')} in {row['family_label']} regions; evidence tier: `{row['evidence_tier']}`."

key_genes = ['SPAG16', 'UBTD2', 'IGHG4', 'IGHA1', 'SCGB1A1', 'SLC39A8']
text = ['## Plain-language result summary', ''] + [gene_line(g) for g in key_genes]
text += [
    '',
    'No gene was independently global-FDR significant in all three chips. SPAG16 is the clearest three-chip directional signal. UBTD2 is concordant across three chips with joint-model support. IPF immunoglobulin remodeling in B-associated regions is mainly K8-supported and patient-sensitive. SCGB1A1 and SLC39A8 are the strongest two-chip AT1-associated signals.'
]
display(Markdown('\n'.join(text)))

In [ ]:
display(Markdown('## Reproducibility information'))
session = {
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'analysis_script': 'scripts/1116_closeout_crosschip_edger_evidence.py',
    'report_source': 'scripts/1116_crosschip_edger_closeout_report.ipynb',
}
display(session)